In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier

X_train = pd.read_csv("../data/X_train_clean.csv", index_col=0)
y_train = pd.read_csv("../data/y_train.csv", index_col=0).squeeze("columns")

best_params = dict(num_leaves=15, n_estimators=300, min_child_samples=100,
                   learning_rate=0.02, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, random_state=42, verbose=-1)

# hold out 25% of the TRAINING data to measure importance
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=42
)

model = LGBMClassifier(**best_params)
model.fit(X_fit, y_fit)

result = permutation_importance(
    model, X_val, y_val, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_train.columns,
    "auc_drop": result.importances_mean.round(4),
    "std": result.importances_std.round(4),
}).sort_values("auc_drop", ascending=False)
print(importance)

                                 feature  auc_drop     std
0   RevolvingUtilizationOfUnsecuredLines    0.0708  0.0024
6                NumberOfTimes90DaysLate    0.0345  0.0009
2   NumberOfTime30-59DaysPastDueNotWorse    0.0313  0.0015
8   NumberOfTime60-89DaysPastDueNotWorse    0.0145  0.0008
1                                    age    0.0076  0.0009
3                              DebtRatio    0.0051  0.0007
5        NumberOfOpenCreditLinesAndLoans    0.0036  0.0004
7           NumberRealEstateLoansOrLines    0.0020  0.0005
4                          MonthlyIncome    0.0011  0.0003
11                           PastDueCode    0.0007  0.0000
9                     NumberOfDependents    0.0001  0.0002
10                         IncomeMissing    0.0001  0.0000
12                             OverLimit    0.0001  0.0001


In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score

late_cols = ["NumberOfTime30-59DaysPastDueNotWorse",
             "NumberOfTime60-89DaysPastDueNotWorse",
             "NumberOfTimes90DaysLate"]

base = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
rng = np.random.default_rng(42)
drops = []
for _ in range(10):
    Xp = X_val.copy()
    Xp[late_cols] = Xp[late_cols].sample(frac=1, random_state=int(rng.integers(1e9))).values
    drops.append(base - roc_auc_score(y_val, model.predict_proba(Xp)[:, 1]))
print(np.mean(drops))

0.11994590858093197


## Which factors drive default?

Method: permutation importance. I fit the tuned LightGBM on 75% of the training data, then, on the other 25% (never seen by the model), shuffle one column at a time and measure how much the AUC falls. A bigger drop means the model relies more on that column. The test set is not used.

Findings (AUC drop):
1. Credit utilization (0.071): the strongest driver, about twice any other column. Borrowers using most of their available credit default far more often.
2. Payment history (0.035 for 90+ days late, 0.031 for 30-59, 0.015 for 60-89): borrowers who paid late before are more likely to default. These three columns overlap, so each looks smaller than the group does.
3. Age (0.008), debt ratio (0.005), open credit lines (0.004), real estate loans (0.002), income (0.001): small but clearly above noise.
4. The flag columns and number of dependents are about 0: OverLimit is redundant with utilization for a tree model, PastDueCode covers only 0.18% of rows, and the others add little.

Caution: this shows what the model relies on, not what causes default.

## Cost of each error type

The dataset has no loan amounts, recoveries or interest rates, so these numbers are assumptions chosen for illustration, not measurements:
- Loan amount (EAD, exposure at default): 10,000
- Loss given default (LGD): 60%, so a defaulted loan loses 6,000
- Profit on a loan that is repaid: 8% of the amount, so 800

The two errors:
- Bad loan approved (borrower defaults): cost = LGD x EAD = 6,000.
- Good loan rejected (borrower would have repaid): cost = the lost profit = 800.

Expected loss of a loan = PD x LGD x EAD, where PD is the model's predicted probability of default. Approve a borrower when the expected loss is smaller than the expected profit:
PD x 6,000 < (1 - PD) x 800, which gives PD < 800 / (6,000 + 800) = 0.118.

This break-even threshold is only valid if the predicted probabilities are roughly correct (calibrated), which I check next.

In [3]:
# assumptions: change these and rerun to see how the answer moves
EAD = 10_000          # loan amount
LGD = 0.60            # share of the loan lost if the borrower defaults
MARGIN = 0.08         # profit on a repaid loan, as a share of the amount

cost_bad_approved = LGD * EAD        # loss from approving a borrower who defaults
cost_good_rejected = MARGIN * EAD    # profit lost by rejecting a borrower who would repay

breakeven = cost_good_rejected / (cost_bad_approved + cost_good_rejected)

print("bad loan approved costs:", cost_bad_approved)
print("good loan rejected costs:", cost_good_rejected)
print("ratio:", round(cost_bad_approved / cost_good_rejected, 2))
print("break-even threshold (approve if PD below):", round(breakeven, 4))

bad loan approved costs: 6000.0
good loan rejected costs: 800.0
ratio: 7.5
break-even threshold (approve if PD below): 0.1176


In [4]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(
    LGBMClassifier(**best_params), X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

print(oof.shape, oof.mean(), y_train.mean())

(120000,) 0.06681831240122886 0.06684166666666666


In [5]:
cal = pd.DataFrame({"pd": oof, "default": y_train.values})
cal["bin"] = pd.qcut(cal["pd"], 10, duplicates="drop")
print(cal.groupby("bin", observed=True).agg(
    avg_pd=("pd", "mean"), actual_rate=("default", "mean"), n=("default", "size")
).round(4))

                    avg_pd  actual_rate      n
bin                                           
(0.00345, 0.00708]  0.0061       0.0031  12000
(0.00708, 0.00919]  0.0081       0.0067  12000
(0.00919, 0.0117]   0.0104       0.0098  12000
(0.0117, 0.0155]    0.0135       0.0116  12000
(0.0155, 0.0222]    0.0185       0.0175  12000
(0.0222, 0.0334]    0.0275       0.0258  12000
(0.0334, 0.0516]    0.0416       0.0405  12000
(0.0516, 0.0841]    0.0661       0.0698  12000
(0.0841, 0.17]      0.1161       0.1160  12000
(0.17, 0.848]       0.3603       0.3677  12000


In [6]:
is_default = (y_train.values == 1)

rows = []
for t in [0.02, 0.04, 0.06, 0.08, 0.10, 0.1176, 0.14, 0.17, 0.20, 0.30, 0.50, 1.01]:
    approve = oof < t
    good_approved = (approve & ~is_default).sum()
    bad_approved = (approve & is_default).sum()
    profit = good_approved * cost_good_rejected - bad_approved * cost_bad_approved
    rows.append({
        "threshold": t,
        "approved_%": round(approve.mean() * 100, 1),
        "default_rate_of_approved_%": round(bad_approved / max(approve.sum(), 1) * 100, 2),
        "profit_per_borrower": round(profit / len(oof), 1),
    })

print(pd.DataFrame(rows))

    threshold  approved_%  default_rate_of_approved_%  profit_per_borrower
0      0.0200        47.3                        0.89                349.9
1      0.0400        64.3                        1.40                453.1
2      0.0600        73.3                        1.80                496.7
3      0.0800        79.1                        2.22                513.0
4      0.1000        83.5                        2.57                521.8
5      0.1176        85.9                        2.81                522.9
6      0.1400        87.9                        3.04                521.5
7      0.1700        90.0                        3.34                515.6
8      0.2000        91.5                        3.60                507.5
9      0.3000        94.3                        4.29                479.4
10     0.5000        98.0                        5.57                412.8
11     1.0100       100.0                        6.68                345.5


In [7]:
def profit_at(t, ead=EAD, lgd=LGD, margin=MARGIN):
    approve = oof < t
    good = (approve & ~is_default).sum()
    bad = (approve & is_default).sum()
    return (good * margin * ead - bad * lgd * ead) / len(oof)

grid = np.arange(0.02, 0.50, 0.005)
rows = []
for lgd in [0.4, 0.6, 0.8]:
    for margin in [0.05, 0.08, 0.12]:
        profits = [profit_at(t, lgd=lgd, margin=margin) for t in grid]
        i = int(np.argmax(profits))
        rows.append({
            "LGD": lgd, "margin": margin,
            "theory_threshold": round(margin / (lgd + margin), 3),
            "best_threshold": round(grid[i], 3),
            "model_profit": round(profits[i], 1),
            "approve_all": round(profit_at(1.01, lgd=lgd, margin=margin), 1),
        })
print(pd.DataFrame(rows))

   LGD  margin  theory_threshold  best_threshold  model_profit  approve_all
0  0.4    0.05             0.111           0.105         321.2        199.2
1  0.4    0.08             0.167           0.155         575.8        479.2
2  0.4    0.12             0.231           0.225         926.8        852.4
3  0.6    0.05             0.077           0.070         282.2         65.5
4  0.6    0.08             0.118           0.115         522.9        345.5
5  0.6    0.12             0.167           0.155         863.7        718.7
6  0.8    0.05             0.059           0.060         254.5        -68.2
7  0.8    0.08             0.091           0.095         479.3        211.8
8  0.8    0.12             0.130           0.130         809.3        585.1


## Does the answer depend on my assumptions?

The costs (LGD, margin) are assumptions, so I repeated the analysis for LGD in {0.4, 0.6, 0.8} and margin in {5%, 8%, 12%}. On out-of-fold predictions for the training data, I searched for the threshold with the highest profit and compared it with the break-even formula margin / (LGD + margin).

Findings:
- The best threshold found by search is within 0.012 of the formula in all 9 scenarios, and the profit curve is flat near the peak, so the formula is enough to choose a threshold.
- The model earns more than approving everyone in all 9 scenarios, by 74 to 323 per applicant. The gain is largest when defaults are costly relative to the margin; with LGD 0.8 and margin 5%, approving everyone loses money (-68 per applicant) while the model earns +255.
- Limits: the costs are assumptions; the margin is treated as a flat share of the loan; every borrower gets the same loan amount.

In [8]:
X_test = pd.read_csv("../data/X_test_clean.csv", index_col=0)
y_test = pd.read_csv("../data/y_test.csv", index_col=0).squeeze("columns")

final = LGBMClassifier(**best_params).fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]

T = breakeven
approve = p_test < T
defaulted = (y_test.values == 1)

good_ap = (approve & ~defaulted).sum()
bad_ap = (approve & defaulted).sum()
profit = (good_ap * cost_good_rejected - bad_ap * cost_bad_approved) / len(y_test)
profit_all = ((~defaulted).sum() * cost_good_rejected - defaulted.sum() * cost_bad_approved) / len(y_test)

print("threshold:", round(T, 4))
print("approved %:", round(approve.mean() * 100, 1))
print("default rate of approved %:", round(bad_ap / approve.sum() * 100, 2))
print("profit per applicant, model:", round(profit, 1))
print("profit per applicant, approve all:", round(profit_all, 1))

threshold: 0.1176
approved %: 85.9
default rate of approved %: 2.78
profit per applicant, model: 525.3
profit per applicant, approve all: 345.5


## Final check on the test set

The threshold (0.1176) was fixed from the cost assumptions before looking at the test set. I refit the tuned LightGBM on all training data and applied the rule to the 30,000 test borrowers, once.

Result: 85.9% of applicants approved; default rate of approved loans 2.78% (6.7% overall); profit per applicant 525.3 against 345.5 for approving everyone, a gain of 179.8 (+52%). This matches the out-of-fold estimate (522.9), so the result is not an artifact of choosing the threshold on training data.